# Phase 14B — NoSQL Generator Fine-tuning (Qwen2.5-Coder-7B-Instruct)

Fine-tunes `Qwen2.5-Coder-7B-Instruct` via LoRA to generate **MongoDB aggregation
pipelines** given enriched schema + key fields + SAR examples. Mirrors Phase 14A
(SQL) but for the NoSQL track, and **warm-starts from the 14A SQL checkpoint**
(the SQL adapter is merged into the base, then a fresh LoRA is trained on MQL).

## Output format the model learns
```json
{"collection": "singer", "pipeline": [{"$match": {"country": "France"}}, {"$count": "total"}]}
```

## Prompt format
```
System: You are an expert MongoDB query writer...

User:
## Database Schema
# Collection: singer
[(singer_id:INT, ...), (name:TEXT, ...), (country:TEXT, ...)]

## Key Fields
singer.country

## Similar Examples
Example 1:
Q: How many singers are older than 25?
MQL: {"collection": "singer", "pipeline": [{"$match": {"age": {"$gt": 25}}}, {"$count": "n"}]}

## Question
How many singers are from France?

Assistant: {"collection": "singer", "pipeline": [{"$match": {"country": "France"}}, {"$count": "total"}]}
```

## Prerequisites (on Drive)
- Phase 12B ✅: `sar_nosql/sar_model.pt`
- Phase 14A ✅: `checkpoints/generator_sql/` (the adapter we warm-start from)
- `nosql_generator_train.jsonl` uploaded to `MyDrive/codegen/generator_data/`
  (built locally in Phase 14B — Cell 4 skips the rebuild if present)

> **Requires A100.** Warm-start merges the SQL adapter into full-precision
> weights, which only works on the bf16 path (on T4 it is skipped and NoSQL
> trains from base). ~45–60 min full run, checkpoint 1 in ~15–20 min.

## Cell 1 — Mount Drive and set paths

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE     = '/content/drive/MyDrive/codegen'
SAR_MODEL      = f'{DRIVE_BASE}/checkpoints/sar_nosql/sar_model.pt'
CHROMA_DIR     = f'{DRIVE_BASE}/indexes/chroma_nosql'
INIT_FROM      = f'{DRIVE_BASE}/checkpoints/generator_sql'      # 14A SQL adapter (warm-start)
GEN_OUT_DIR    = f'{DRIVE_BASE}/checkpoints/generator_nosql'
GEN_DATA_DIR   = f'{DRIVE_BASE}/generator_data'
GEN_DATA_FILE  = f'{GEN_DATA_DIR}/nosql_generator_train.jsonl'

os.makedirs(GEN_OUT_DIR,  exist_ok=True)
os.makedirs(GEN_DATA_DIR, exist_ok=True)

for path, label in [
    (SAR_MODEL,  'SAR NoSQL model'),
    (INIT_FROM,  '14A SQL checkpoint (warm-start)'),
    (GEN_DATA_FILE, 'NoSQL training data'),
]:
    exists = os.path.exists(path)
    size   = f'{os.path.getsize(path)/1e6:.1f} MB' if exists and os.path.isfile(path) else ''
    print(f'{label}: {"✅" if exists else "❌"} {size}')

import torch
print(f'\nGPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')
USE_A100 = torch.cuda.is_available() and 'A100' in torch.cuda.get_device_name(0)
print(f'A100 mode: {USE_A100}   (warm-start requires A100)')

## Cell 2 — Clone / update repo

In [ ]:
%%bash
set -e
REPO="/content/Codegen"
BRANCH="phase/14b-generator-nosql"

if [ -d "$REPO/.git" ]; then
    cd "$REPO" && git fetch origin && git checkout $BRANCH && git pull origin $BRANCH
else
    git clone https://github.com/kethansplunk/Codegen.git "$REPO"
    cd "$REPO" && git checkout $BRANCH
fi
echo "Branch : $(git branch --show-current)"
echo "Commit : $(git log --oneline -1)"

## Cell 3 — Install dependencies

Plain `transformers.Trainer` (no TRL). torchao is handled inside `train.py`.

In [ ]:
!pip install -q chromadb "FlagEmbedding==1.2.9" peft bitsandbytes accelerate datasets

import transformers
from transformers import Trainer, DataCollatorForSeq2Seq
from peft import prepare_model_for_kbit_training, get_peft_model, PeftModel
print(f'transformers {transformers.__version__} ✅  (Trainer, no TRL)')
print('Dependencies installed ✅')

## Cell 4 — Build NoSQL generator training data

Skips if `nosql_generator_train.jsonl` is already on Drive (recommended: build
it locally and upload, like Phase 14A). Building here needs the NoSQL ChromaDB
index on Drive.

In [ ]:
import os, sys, subprocess
sys.path.insert(0, '/content/Codegen')

if os.path.exists(GEN_DATA_FILE):
    with open(GEN_DATA_FILE) as f:
        n = sum(1 for _ in f)
    print(f'Training data already exists: {n} entries — skipping build ✅')
else:
    cmd = [
        'python', '-m', 'scripts.build_generator_training_data', '--track', 'nosql',
        '--cot',        '/content/Codegen/Data/cot_data/nosql_cot_train.json',
        '--chroma_dir', CHROMA_DIR,
        '--sar_model',  SAR_MODEL,
        '--out',        GEN_DATA_FILE,
    ]
    if subprocess.run(cmd, cwd='/content/Codegen').returncode != 0:
        raise RuntimeError('Training data build failed')

## Cell 5 — Inspect training data

In [ ]:
import json
with open(GEN_DATA_FILE) as f:
    samples = [json.loads(line) for line in f]
print(f'Training examples: {len(samples)}')
print('\n=== Sample entry (truncated) ===')
print(samples[0]['text'][:1200])
print('...')

## Cell 6 — Fine-tune NoSQL Generator (warm-started from 14A)

`--init_from` merges the 14A SQL adapter into the base weights, then a fresh
LoRA is trained on the MQL data. Same memory-safe config as 14A (gradient
checkpointing, batch 2 × accum 8, seq 2048). Live `[TRAIN] %` + `[CKPT]` output.

| | value |
|---|---|
| Steps/epoch | ~ceil(5410 / 16) = 339 |
| Total steps (3 epochs) | ~1017 |
| Checkpoint 1 | end of epoch 1 (~step 339, ~15–20 min) |
| Full run | ~40–55 min |

In [ ]:
import os, subprocess

REPO = '/content/Codegen'
os.makedirs(f'{REPO}/Data/generator_data', exist_ok=True)
link = f'{REPO}/Data/generator_data/nosql_generator_train.jsonl'
if not os.path.exists(link):
    os.symlink(GEN_DATA_FILE, link)

cmd = [
    'python', '-u', '-m', 'src.generator.train',
    '--data', GEN_DATA_FILE,
    '--out',  GEN_OUT_DIR,
]
if USE_A100:
    cmd.append('--use_a100')
    cmd += ['--init_from', INIT_FROM]     # warm-start from 14A (bf16 only)

print('Running:', ' '.join(cmd), '\n', flush=True)

proc = subprocess.Popen(cmd, cwd=REPO, text=True,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
tail = []
for line in proc.stdout:
    print(line, end='', flush=True)
    tail.append(line); tail = tail[-40:]
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f'Training failed (exit {proc.returncode}). Last {len(tail)} lines:\n' + ''.join(tail))
print('\nTraining complete ✅')

## Cell 7 — Verify saved checkpoint

In [ ]:
import os
print(f'Checkpoint directory: {GEN_OUT_DIR}')
files = os.listdir(GEN_OUT_DIR) if os.path.exists(GEN_OUT_DIR) else []
total_mb = sum(os.path.getsize(os.path.join(dp, f))
               for dp, _, fs in os.walk(GEN_OUT_DIR) for f in fs) / 1e6 if files else 0
print(f'Files  : {len(files)}')
print(f'Size   : {total_mb:.1f} MB')
for fname in sorted(files):
    print(f'  {fname}')

## Cell 8 — Smoke test: generate MQL from the fine-tuned model

Loads the trained adapter and generates MongoDB pipelines for a few questions,
using static SAR-style examples (no FlagEmbedding / ChromaDB needed — the full
retriever pipeline is exercised in Phase 16).

In [ ]:
import sys
sys.path.insert(0, '/content/Codegen')
from src.generator.infer import GeneratorInfer

# track='nosql' selects the MongoDB system prompt + MQL example format that the
# 14B adapter was trained with (must match training, else output degrades).
gen = GeneratorInfer(GEN_OUT_DIR, track='nosql', n_candidates=1, temperature=0.0)

schema = '''# Collection: singer
[
(singer_id:INT, Primary Key),
(name:TEXT, Examples: [Ed Sheeran, Adele]),
(country:TEXT, Examples: [France, UK, USA]),
(age:INT, Examples: [25, 32, 40]),
]'''
key_fields = ['singer.country', 'singer.age']

# Static SAR-style NoSQL examples (question + mql_collection + mql_pipeline).
examples = [
    {'question': 'How many singers are there in each country?',
     'mql_collection': 'singer',
     'mql_pipeline': [{'$group': {'_id': '$country', 'n': {'$sum': 1}}}]},
    {'question': 'How many singers are older than 25?',
     'mql_collection': 'singer',
     'mql_pipeline': [{'$match': {'age': {'$gt': 25}}}, {'$count': 'n'}]},
]

for q in ['How many singers are from each country?',
          'How many singers are older than 30?',
          'List the names of singers from France.']:
    print('Q:  ', q)
    print('MQL:', gen.generate(q, schema, key_fields, examples)[0], '\n')